## Working through Dataset trying it without AI

In [188]:
import pandas as pd
import numpy as np
df_x = pd.read_excel("data/Data Dictionary.xls")
df_t = pd.read_csv("data/cs-training.csv")
df_ts = pd.read_csv("data/cs-test.csv")

In [189]:
set(df_t.columns.to_list()).difference(set(df_ts.columns.to_list()))
df_t.shape

(150000, 12)

In [190]:
df_t.info()
df_t.describe()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [191]:
df_x.info()
df_x.describe()
df_x

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  12 non-null     str  
 1   Unnamed: 1  12 non-null     str  
 2   Unnamed: 2  12 non-null     str  
dtypes: str(3)
memory usage: 420.0 bytes


,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,Variable Name,Description,Type
1,SeriousDlqin2yrs,Person experienced 90 days past due delinquenc...,Y/N
2,RevolvingUtilizationOfUnsecuredLines,Total balance on credit cards and personal lin...,percentage
3,age,Age of borrower in years,integer
4,NumberOfTime30-59DaysPastDueNotWorse,Number of times borrower has been 30-59 days p...,integer
5,DebtRatio,"Monthly debt payments, alimony,living costs di...",percentage
6,MonthlyIncome,Monthly income,real
7,NumberOfOpenCreditLinesAndLoans,Number of Open loans (installment like car loa...,integer
8,NumberOfTimes90DaysLate,Number of times borrower has been 90 days or m...,integer
9,NumberRealEstateLoansOrLines,Number of mortgage and real estate loans inclu...,integer


### Missing Values

In [192]:
df = df_t.copy()
df["MonthlyIncome"] = df_t["MonthlyIncome"].fillna(df_t["MonthlyIncome"].median())
df["NumberOfDependents"] = df_t["NumberOfDependents"].fillna(df_t["NumberOfDependents"].median()).astype("int")

### Feature engineering
Change heavy skewed cols into log

In [193]:
df["log_DebtRatio"] = np.log(df_t["DebtRatio"] + 1e-9)
print("before:", df["DebtRatio"].mean())
print("after:", df["log_DebtRatio"].mean())

def feature_engineer(X):
    x = X.copy()
    x["log_DebtRatio"] = np.log(x["DebtRatio"] + 1e-9)
    x["log_MonthlyIncome"] = np.log(x["MonthlyIncome"] + 1e-9)
    return x

before: 353.00507576386985
after: -0.46714511735734715


### Pipelines

In [194]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import (Pipeline, FunctionTransformer)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (Normalizer, StandardScaler)

num_cols = df.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"]).columns.to_list()
nump = Pipeline([("imp", SimpleImputer(strategy="median")), ("stand", StandardScaler()), ("norm", Normalizer())])

featf = FunctionTransformer(feature_engineer)
ct = ColumnTransformer([("nums", nump, num_cols)])
pre = Pipeline([("features", featf), ("ct", ct)])
ct

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nums', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{f

### Logistic Baseline

In [195]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = Pipeline([("pre", pre), ("model", LogisticRegression())])
df = df.drop(columns=["Unnamed: 0"])
y = df["SeriousDlqin2yrs"]; X = df.drop(columns=["SeriousDlqin2yrs"])
model = clf.fit(X, y)
roc_auc = cross_val_score(clf, df.drop(columns=["SeriousDlqin2yrs"]), df["SeriousDlqin2yrs"], cv=5, scoring="roc_auc")
print("Cross val", roc_auc.mean().round(3), "+-", roc_auc.std().round(3))

Cross val 0.807 +- 0.006


In [203]:
predictions = pd.DataFrame()
X_test = df_ts.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"])

# I came up with this!!!
predictions["Probability"] = model.predict_proba(X_test)[:,1]
predictions.index = range(1,len(predictions) + 1)

In [204]:
predictions.shape

(101503, 1)

In [205]:
final = predictions[["Probability"]].reset_index().rename(columns={"index": "Id"})
final.to_csv("data/test_out.csv", index=False)